In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [2]:
# set seaborn style
sns.set_style("white")
sns.set_context("talk")

In [3]:
# set random seed
seed = 42

In [4]:
# import data
df = pd.read_csv("../data/processed/corpus_studio_processed.csv")
df.drop(
    columns=["performance_type", "group"], inplace=True
)  # for this study, only studio recordings were included
df.fillna(
    "", inplace=True
)  # replace NaN values for previous and next words with empty strings
df.head()

,word,previous_word,next_word,artist,song_type,song,aae_feature,aae_realization,time,social_group
0,first,the,thing,muddywaters,cover,rollin_stone,post-consonantal t,1,1960s,AA
1,your,rock,baby,tenyearsafter,original,rock_your_mama,post-vocalic r,1,1960s,nonAA_nonUS
2,for,do,you,bbking,original,please_love_me,post-vocalic r,0,1960s,AA
3,end,wanna,it,robertcray,original,smoking_gun,post-consonantal d,1,1980s,AA
4,serves,it,me,johnleehooker,cover,it_serves_me_right,third person singular,0,1980s,AA


In [5]:
# define target and predictor columns
target_col = "aae_realization"
X = df.loc[:, df.columns != target_col]
y = df.loc[:, target_col]

# define categorical features
cat_features = [
    "previous_word",
    "word",
    "next_word",
    "artist",
    "song_type",
    "song",
    "aae_feature",
    "time",
    "social_group",
]

In [6]:
# create train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

In [7]:
# hyperparameter tuning with optuna
def objective(trial):
    params = {
        "iterations": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.05, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.05, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
    }

    model = CatBoostClassifier(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        cat_features=cat_features,
        verbose=0,
        early_stopping_rounds=100,
    )

    preds = model.predict(X_test)
    pred_labels = np.rint(preds)
    score = f1_score(y_test, pred_labels)
    return score

In [8]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=600)
print("Number of completed trials: {}".format(len(study.trials)))
print("Best trial:")
trial = study.best_trial

print("\tBest Score: {}".format(trial.value))
print("\tBest Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2024-03-04 15:30:28,502] A new study created in memory with name: no-name-cb777627-62ad-46ae-99af-c9974bf617bc


[I 2024-03-04 15:30:32,466] Trial 0 finished with value: 0.912301013024602 and parameters: {'learning_rate': 0.001372193414304402, 'depth': 3, 'subsample': 0.29150646327101054, 'colsample_bylevel': 0.6358485887955089, 'min_data_in_leaf': 29}. Best is trial 0 with value: 0.912301013024602.
[I 2024-03-04 15:30:39,259] Trial 1 finished with value: 0.9307520476545048 and parameters: {'learning_rate': 0.028336195531419674, 'depth': 8, 'subsample': 0.5581679378780366, 'colsample_bylevel': 0.3160011605390697, 'min_data_in_leaf': 53}. Best is trial 1 with value: 0.9307520476545048.
[I 2024-03-04 15:30:46,892] Trial 2 finished with value: 0.9288045090477604 and parameters: {'learning_rate': 0.01653895340739081, 'depth': 7, 'subsample': 0.1110023242715199, 'colsample_bylevel': 0.4240027712807265, 'min_data_in_leaf': 29}. Best is trial 1 with value: 0.9307520476545048.
[I 2024-03-04 15:30:53,802] Trial 3 finished with value: 0.9295690936106984 and parameters: {'learning_rate': 0.00734066678919587

Number of completed trials: 50
Best trial:
	Best Score: 0.9332143388368288
	Best Params: 
    learning_rate: 0.022441846333988966
    depth: 7
    subsample: 0.45687566651080874
    colsample_bylevel: 0.7549348027207174
    min_data_in_leaf: 44


In [9]:
# visualize parameter importance
optuna.visualization.plot_param_importances(study)

In [10]:
# visualize study optimization
optuna.visualization.plot_optimization_history(study)

In [11]:
# final model
final_model = CatBoostClassifier(**trial.params)
final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    cat_features=cat_features,
    verbose=0,
    early_stopping_rounds=100,
)

y_pred = final_model.predict(X_test)
pred_labels = np.rint(y_pred)
score = f1_score(y_test, pred_labels)
print(f"the f1-score of the final model is {score}")

the f1-score of the final model is 0.9332143388368288


In [12]:
# saving model
final_model.save_model(fname="../models/model.json", format="json")